# Notebook 03 — Semantic Model Configuration

Step-by-step guide to creating a Power BI semantic model from the `university` Delta tables in your Fabric Lakehouse. This notebook is primarily documentation with one verification query.

## Step 1: Create Semantic Model from Lakehouse

1. Open your Lakehouse in the Fabric portal
2. Click **New semantic model** (top toolbar)
3. Name it: `University Analytics`
4. Select all 13 tables from the `university` schema:
   - `dim_date`, `dim_department`, `dim_program`, `dim_staff`, `dim_course`
   - `bridge_course_program`, `dim_exam_type`, `dim_fee_type`, `dim_academic_period`
   - `dim_student`, `fact_enrollments`, `fact_exam_results`, `fact_financial_transactions`
5. Click **Confirm**

## Step 2: Configure Relationships

Open the semantic model in the web modelling view and configure these **18 relationships**:

| # | From Table | From Column | To Table | To Column | Cardinality | Filter |
|---|-----------|-------------|----------|-----------|-------------|--------|
| 1 | fact_enrollments | student_key | dim_student | student_key | Many:1 | Single |
| 2 | fact_enrollments | course_key | dim_course | course_key | Many:1 | Single |
| 3 | fact_enrollments | academic_period_key | dim_academic_period | academic_period_key | Many:1 | Single |
| 4 | fact_enrollments | program_key | dim_program | program_key | Many:1 | Single |
| 5 | fact_enrollments | enroll_date_key | dim_date | date_key | Many:1 | Single |
| 6 | fact_exam_results | student_key | dim_student | student_key | Many:1 | Single |
| 7 | fact_exam_results | course_key | dim_course | course_key | Many:1 | Single |
| 8 | fact_exam_results | exam_type_key | dim_exam_type | exam_type_key | Many:1 | Single |
| 9 | fact_exam_results | academic_period_key | dim_academic_period | academic_period_key | Many:1 | Single |
| 10 | fact_exam_results | staff_key | dim_staff | staff_key | Many:1 | Single |
| 11 | fact_financial_transactions | student_key | dim_student | student_key | Many:1 | Single |
| 12 | fact_financial_transactions | fee_type_key | dim_fee_type | fee_type_key | Many:1 | Single |
| 13 | fact_financial_transactions | academic_period_key | dim_academic_period | academic_period_key | Many:1 | Single |
| 14 | dim_program | department_key | dim_department | department_key | Many:1 | Single |
| 15 | dim_staff | department_key | dim_department | department_key | Many:1 | Single |
| 16 | dim_course | department_key | dim_department | department_key | Many:1 | Single |
| 17 | dim_course | coordinator_staff_key | dim_staff | staff_key | Many:1 | Single |
| 18 | dim_student | program_key | dim_program | program_key | Many:1 | Single |

## Step 3: DAX Measures

Create the following measures in the semantic model. Organize them into **Display Folders**.

### Enrolment Analytics

```dax
Total Enrolments = COUNTROWS(fact_enrollments)

Active Enrolments = CALCULATE(COUNTROWS(fact_enrollments), fact_enrollments[enrollment_status] = "Enrolled")

Unique Students = DISTINCTCOUNT(fact_enrollments[student_key])

Completion Rate = DIVIDE(CALCULATE(COUNTROWS(fact_enrollments), fact_enrollments[enrollment_status] = "Completed"), COUNTROWS(fact_enrollments))

Withdrawal Rate = DIVIDE(CALCULATE(COUNTROWS(fact_enrollments), fact_enrollments[enrollment_status] = "Withdrawn"), COUNTROWS(fact_enrollments))

Total MCs Earned = SUM(fact_enrollments[credit_points_earned])

Total MCs Attempted = SUM(fact_enrollments[credit_points_attempted])
```

### Academic Performance

```dax
Average Score = AVERAGE(fact_exam_results[score_percentage])

Average GPA = AVERAGE(fact_exam_results[grade_points])

Pass Rate = DIVIDE(CALCULATE(COUNTROWS(fact_exam_results), fact_exam_results[grade_letter] <> "F"), COUNTROWS(fact_exam_results))

Distinction Rate = DIVIDE(CALCULATE(COUNTROWS(fact_exam_results), fact_exam_results[grade_letter] IN {"A+", "A", "A-"}), COUNTROWS(fact_exam_results))

Exam Count = COUNTROWS(fact_exam_results)

Average Weighted Score = AVERAGE(fact_exam_results[weighted_score])

Fail Count = CALCULATE(COUNTROWS(fact_exam_results), fact_exam_results[grade_letter] = "F")
```

### Financial Analytics

```dax
Total Revenue = CALCULATE(SUM(fact_financial_transactions[amount]), fact_financial_transactions[transaction_type] = "Charge")

Total Payments = CALCULATE(SUM(fact_financial_transactions[amount]), fact_financial_transactions[transaction_type] = "Payment")

Outstanding Balance = [Total Revenue] - [Total Payments]

Total Scholarships = CALCULATE(SUM(fact_financial_transactions[amount]), fact_financial_transactions[transaction_type] = "Credit")

Transaction Count = COUNTROWS(fact_financial_transactions)

Average Tuition = CALCULATE(AVERAGE(fact_financial_transactions[amount]), fact_financial_transactions[transaction_type] = "Charge", dim_fee_type[fee_type_name] = "Tuition")

Overdue Amount = CALCULATE(SUM(fact_financial_transactions[amount]), fact_financial_transactions[is_overdue] = TRUE())
```

### Per-Student Metrics

```dax
Revenue Per Student = DIVIDE([Total Revenue], [Unique Students])

MCs Per Student = DIVIDE([Total MCs Earned], [Unique Students])

GPA Per Student = AVERAGEX(VALUES(dim_student[student_key]), CALCULATE(AVERAGE(fact_exam_results[grade_points])))

Avg Exams Per Student = DIVIDE([Exam Count], [Unique Students])

Avg Courses Per Student = DIVIDE([Total Enrolments], [Unique Students])

Payment Rate = DIVIDE([Total Payments], [Total Revenue])

Scholarship Rate = DIVIDE(CALCULATE(COUNTROWS(dim_student), dim_student[scholarship_flag] = TRUE()), COUNTROWS(dim_student))
```

## Step 4: Hierarchies

Create these hierarchies in the semantic model:

| Hierarchy | Levels |
|-----------|--------|
| Academic Calendar | `dim_academic_period[academic_year]` → `dim_academic_period[semester]` |
| Academic Structure | `dim_department[faculty]` → `dim_department[department_name]` → `dim_program[program_name]` |
| Course Hierarchy | `dim_course[level]` → `dim_course[course_name]` |
| Fee Hierarchy | `dim_fee_type[fee_category]` → `dim_fee_type[fee_type_name]` |

## Step 5: Row-Level Security (RLS)

Configure two RLS roles in the semantic model:

### Staff Role
- **Name:** `Staff`
- **Filter:** None (full access to all data)

### Student Role
- **Name:** `Student`
- **DAX Filter on `dim_student`:**
```dax
[email] = USERPRINCIPALNAME()
```
This ensures each student can only see their own enrolment, exam, and financial data.

In [ ]:
# Verification: confirm table counts match expected
expected = {
    "dim_date": 3287, "dim_department": 8, "dim_program": 12,
    "dim_staff": 80, "dim_course": 60, "bridge_course_program": 380,
    "dim_exam_type": 7, "dim_fee_type": 8, "dim_academic_period": 8,
    "dim_student": 520,
}

for tbl, exp in expected.items():
    actual = spark.table(f'university.{tbl}').count()
    status = 'PASS' if actual == exp else 'WARN'
    print(f"  [{status}] {tbl}: {actual:,} rows (expected {exp:,})")

# Fact tables have variable counts
for tbl in ["fact_enrollments", "fact_exam_results", "fact_financial_transactions"]:
    actual = spark.table(f'university.{tbl}').count()
    print(f"  [INFO] {tbl}: {actual:,} rows")

## Next Steps

Proceed to **Notebook 04** for the Copilot demo with scripted prompts and verification queries.